# Current Stores - Bronze Layer

Transforms raw store location data into standardized format with dummy annual sales data.

**Input**: Raw locations table (manually uploaded)

**Output**: `{catalog}.{bronze_schema}.current_stores_ne` - Standardized Northeast store locations with sales data

**Supported States**: MA, CT, NJ, MD (Northeast region)

## Parameters

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when
from datetime import datetime

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("lce_locations_table", "")

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
lce_locations_table = dbutils.widgets.get("lce_locations_table")

assert catalog and bronze_schema and lce_locations_table, "Missing required parameters: catalog, bronze_schema, lce_locations_table"

# Define output table
output_table = f"{catalog}.{bronze_schema}.current_stores_ne"

print(f"Catalog: {catalog}")
print(f"Bronze schema: {bronze_schema}")
print(f"Input table: {lce_locations_table}")
print(f"Output table: {output_table}")

## Validate Input Table

In [ ]:
# Validate input table exists
try:
    input_df = spark.table(lce_locations_table)
    input_count = input_df.count()
    print(f"✓ Input table found: {lce_locations_table}")
    print(f"  Total records: {input_count:,}")
except Exception as e:
    print(f"\n❌ ERROR: Input table not found: {lce_locations_table}")
    print(f"\nPlease ensure the locations table has been manually uploaded.")
    print(f"\nExpected table: {lce_locations_table}")
    print(f"Required columns: location_id, store_name, latitude, longitude, address, city, state, zip_code")
    raise RuntimeError(f"Input table not found: {lce_locations_table}") from e

# Validate required columns
required_columns = ['location_id', 'latitude', 'longitude', 'city', 'state']
missing_columns = [c for c in required_columns if c not in input_df.columns]

if missing_columns:
    print(f"\n❌ ERROR: Missing required columns: {missing_columns}")
    print(f"\nAvailable columns: {input_df.columns}")
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"\n✓ All required columns present")
print(f"\nInput schema:")
input_df.printSchema()

# Show state distribution
print("\nStores by state:")
display(input_df.groupBy("state").count().orderBy("state"))

## Create Standardized Current Stores Table with Sales Data

In [ ]:
# Build standardized DataFrame
# Handle optional columns gracefully
available_columns = input_df.columns

stores_df = input_df.select(
    col("location_id").cast("int"),
    col("store_name") if "store_name" in available_columns else lit(None).alias("store_name"),
    col("latitude").cast("double"),
    col("longitude").cast("double"),
    col("address") if "address" in available_columns else lit(None).alias("address"),
    col("city"),
    col("state"),
    col("zip_code") if "zip_code" in available_columns else lit(None).alias("zip_code")
)

# Add standardized columns
stores_df = stores_df.withColumn("store_type", lit("Current Store")) \
    .withColumn("country_code", lit("US"))

# Generate realistic dummy annual sales ($300K-$800K range)
# Variation based on hash of location_id for reproducibility
stores_df = stores_df.withColumn(
    "annual_sales",
    (lit(300000) + (F.abs(F.hash(col("location_id"))) % 500000)).cast("bigint")
)

# Calculate monthly sales
stores_df = stores_df.withColumn(
    "monthly_sales",
    (col("annual_sales") / 12).cast("bigint")
)

# Add ingestion timestamp
stores_df = stores_df.withColumn("ingestion_timestamp", F.current_timestamp())

# Filter out records with missing coordinates
stores_df = stores_df.filter(
    col("latitude").isNotNull() & col("longitude").isNotNull()
)

print(f"Processed {stores_df.count()} stores with valid coordinates")
display(stores_df.limit(5))

In [ ]:
# Write to Delta table
(
    stores_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

# Add table comment
spark.sql(f"""
    COMMENT ON TABLE {output_table} IS 
    'Store locations in Northeast region (MA, CT, NJ, MD) with dummy annual sales data'
""")

print(f"✓ Created table: {output_table}")

## Validation

In [ ]:
print("=" * 80)
print("CURRENT STORES VALIDATION")
print("=" * 80)

# Summary statistics
summary = spark.sql(f"""
    SELECT 
        COUNT(*) as total_stores,
        COUNT(DISTINCT location_id) as unique_locations,
        COUNT(CASE WHEN latitude IS NOT NULL AND longitude IS NOT NULL THEN 1 END) as stores_with_coords,
        COUNT(DISTINCT city) as unique_cities,
        COUNT(DISTINCT state) as unique_states,
        ROUND(AVG(annual_sales), 0) as avg_annual_sales,
        ROUND(MIN(annual_sales), 0) as min_annual_sales,
        ROUND(MAX(annual_sales), 0) as max_annual_sales
    FROM {output_table}
""")
display(summary)

# Stores by state
print("\nStores by state:")
display(spark.sql(f"""
    SELECT 
        state,
        COUNT(*) as store_count,
        ROUND(AVG(annual_sales), 0) as avg_annual_sales
    FROM {output_table}
    GROUP BY state
    ORDER BY store_count DESC
"""))

# Sample records
print("\nSample store locations:")
display(spark.table(output_table).select(
    "location_id", "store_name", "store_type", "city", "state", 
    "latitude", "longitude", "country_code", "annual_sales"
).limit(10))

# Validate all coordinates are present
missing_coords = spark.sql(f"""
    SELECT COUNT(*) as missing_count
    FROM {output_table}
    WHERE latitude IS NULL OR longitude IS NULL
""").collect()[0]['missing_count']

if missing_coords > 0:
    print(f"\n⚠️  WARNING: {missing_coords} stores missing coordinates")
else:
    print(f"\n✓ All stores have valid coordinates")

# Validate sales data
missing_sales = spark.sql(f"""
    SELECT COUNT(*) as missing_count
    FROM {output_table}
    WHERE annual_sales IS NULL OR monthly_sales IS NULL
""").collect()[0]['missing_count']

if missing_sales > 0:
    print(f"\n⚠️  WARNING: {missing_sales} stores missing sales data")
else:
    print(f"\n✓ All stores have valid sales data")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)